# E6b — Preservation vs Destruction (Phase 10, W-lane flight)

**What this does**: flies the E6b preservation battery (straight-heavy,
U/S-enriched: U 160 items/136 straight ≈5.7x E6, S 28 needles x 4 fills =
96 straight rows ≈6.4x, F 40 as replication block, T dropped) on
Qwen2.5-1.5B-Instruct under three conditions — **base**, **+adapter_real**,
**+adapter_scrambled** — to power the E6 straight-subset pattern properly:
does coherent geometry PRESERVE native report-relevant signal that
scrambled geometry destroys?

**Pre-registered structure** (docs/E6B_PROTOCOL.md, locked before flight):
PRIMARY = joint J = Δρ_straight_U + Δρ_straight_S (real − scrambled) under
simultaneous item-level condition-label permutation, α=.05; arm-level
localization Holm-corrected (.025/.05). Preservation established ⇔ joint
p<.05 AND both arm deltas positive.

Additions outside the battery (never pooled): **precheck v2** — 26 pairs
(6 wing + 10 held-out synonym-band ≤2.8° + 10 held-out opposition-band
≥94°); the off-mean bands discriminate real instillation from scrambled
scale-matching (E6 precheck limit). **Catch trials x10 quantities**
(E6's 6 verbatim + 4 new; original-6 reported separately for comparability).

Inputs from Drive: battery `gdrive:semcore/e6b/e6b_battery.json`, pack
`gdrive:semcore/e4/e4_dictionary_pack.json`, adapters = newest
`{arm}_full_*` under `gdrive:semcore/e4/`. Results → `gdrive:semcore/e6b/`
(per-condition inflight shipping — the outage law).

SMOKE mode: `/content/SMOKE` present → real condition only, ~6 items/arm,
2 fills (incl. the new 0.55 level), K=4. Never pooled.


In [ ]:
# ── Setup: GPU, installs, rclone, inputs, conditions ─────────────────────────
import subprocess, sys, os, json, re, math, time
from pathlib import Path
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # set before CUDA init

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')

print('Installing packages...')
subprocess.run([sys.executable,'-m','pip','uninstall','-q','-y','torchao'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
    'transformers>=4.44','peft>=0.11','accelerate','scipy','pandas'], check=True)

import torch
assert torch.cuda.is_available(), 'No GPU — request a T4.'
DEV = 'cuda'

if subprocess.run(['which','rclone'], capture_output=True).returncode != 0:
    subprocess.run('curl -s https://rclone.org/install.sh | bash', shell=True, capture_output=True)
RCLONE_CONF = '/content/rclone.conf'
HAS_RCLONE = os.path.exists(RCLONE_CONF)
print('rclone conf:', 'present' if HAS_RCLONE else 'MISSING')

def rc(*args, check=True, capture=False):
    cmd = ['rclone','--config',RCLONE_CONF] + list(args)
    return subprocess.run(cmd, check=check, capture_output=capture, text=True)

BATTERY = Path('/content/e6b_battery.json')
if not BATTERY.exists() and HAS_RCLONE:
    rc('copy','gdrive:semcore/e6b/e6b_battery.json','/content/')
battery = json.load(open(BATTERY))
print('battery:', battery['name'], 'v'+battery['version'])

PACK = Path('/content/e4_dictionary_pack.json')
if not PACK.exists() and HAS_RCLONE:
    rc('copy','gdrive:semcore/e4/e4_dictionary_pack.json','/content/')
pack = json.load(open(PACK))
print('pack:', pack['name'], '| concepts', pack['n_concepts'])

SMOKE = Path('/content/SMOKE').exists()
print('MODE:', 'SMOKE' if SMOKE else 'FULL')

# discover newest adapter dir per arm on Drive (probe-fix pattern)
ADAPTERS = {}
if HAS_RCLONE:
    lsd = rc('lsd','gdrive:semcore/e4/', capture=True).stdout
    dirs = [l.split()[-1] for l in lsd.strip().splitlines() if l.strip()]
    for arm in ('real','scrambled'):
        cand = sorted(d for d in dirs if d.startswith(f'{arm}_full_'))
        if cand:
            src = f'gdrive:semcore/e4/{cand[-1]}/adapter_{arm}'
            dst = f'/content/adapter_{arm}'
            rc('copy', src, dst, check=False)
            if Path(dst, 'adapter_config.json').exists():
                ADAPTERS[arm] = dst
                print(f'adapter {arm}: {cand[-1]}')

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
if SMOKE:
    CONDITIONS = ['real']
    assert 'real' in ADAPTERS, 'smoke needs the real adapter'
else:
    CONDITIONS = ['base', 'real', 'scrambled']
    assert 'real' in ADAPTERS and 'scrambled' in ADAPTERS, \
        'full flight needs BOTH adapters shipped on Drive'
COND_ADAPTER = {'base': None,
                'real': ADAPTERS.get('real'),
                'scrambled': ADAPTERS.get('scrambled')}
print('conditions:', CONDITIONS)

OUT = Path('/content/e6b_out'); OUT.mkdir(exist_ok=True)
SEED = 20260822
torch.manual_seed(SEED)
STAMP = time.strftime('%Y%m%d_%H%M', time.gmtime())   # one stamp for inflight + final dirs

K_SAMPLES_U = 4 if SMOKE else 8
FILL_FRACTIONS = [0.05, 0.55] if SMOKE else battery['fills']
EFFECTIVE_WINDOW_CAP = 8192   # T4 law from E5 smoke-1 OOM


GPU: Tesla T4, 15360 MiB
Installing packages...


rclone conf: present


battery: E6b preservation battery v1.0


pack: E4 dictionary pack | concepts 3052
MODE: SMOKE


adapter real: real_full_20260821_2144


adapter scrambled: scrambled_full_20260821_2221
conditions: ['real']


In [ ]:
# ── Battery prep: smoke subsetting; polarity comes FROM the battery ──────────
import copy
bat = copy.deepcopy(battery['arms'])

for name, arm in bat.items():   # polarity is pre-registered in the file
    assert all('flipped' in it for it in arm['items']), f'{name}: missing flipped field'

def subset(items, keep_ids):
    return [it for it in items if it['id'] in keep_ids]

if SMOKE:
    bat['uncertainty']['items'] = subset(bat['uncertainty']['items'],
        {'U49','U53','U102','U106','U155','U159'})   # 1 straight + 1 flipped per band
    bat['familiarity']['items'] = subset(bat['familiarity']['items'],
        {'F41','F43','F48','F53','F61','F70','F76','F79'})
    bat['saturation']['items'] = subset(bat['saturation']['items'], {'S11','S15'})

for name, arm in bat.items():
    ns = sum(1 for it in arm['items'] if not it['flipped'])
    print(f"{name}: {len(arm['items'])} items ({ns} straight)")


uncertainty: 6 items (3 straight)
familiarity: 8 items (5 straight)
saturation: 2 items (1 straight)


In [ ]:
# ── Harness (E5/E6 verbatim + optional adapter; keyword input_ids for PEFT) ──
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

INT_RE = re.compile(r'\b(10|[0-9])\b')

class Harness:
    def __init__(self, model_id, adapter_path=None, label=None):
        self.model_id = model_id
        self.short = label or model_id.split('/')[-1]
        self.tok = AutoTokenizer.from_pretrained(model_id)
        base = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, device_map=DEV)
        cfg_ctx = getattr(base.config, 'max_position_embeddings', 8192)
        self.model = PeftModel.from_pretrained(base, adapter_path) if adapter_path else base
        self.model.eval()
        self.window = min(cfg_ctx, EFFECTIVE_WINDOW_CAP)
        print(f'{self.short}: window={self.window} (config {cfg_ctx})'
              + (f' adapter={adapter_path}' if adapter_path else ' [no adapter]'))

    def chat_ids(self, user, system=None):
        msgs = ([{'role':'system','content':system}] if system else []) + \
               [{'role':'user','content':user}]
        text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return self.tok(text, return_tensors='pt').input_ids.to(DEV)

    @torch.no_grad()
    def greedy(self, user, system=None, max_new=32, with_stats=False):
        ids = self.chat_ids(user, system)
        out = self.model.generate(input_ids=ids, max_new_tokens=max_new, do_sample=False,
                                  output_scores=with_stats, return_dict_in_generate=True,
                                  pad_token_id=self.tok.eos_token_id)
        text = self.tok.decode(out.sequences[0, ids.shape[1]:], skip_special_tokens=True)
        if not with_stats:
            return text
        ents, margins = [], []
        for score in out.scores:
            p = torch.softmax(score[0].float(), dim=-1)
            ents.append(float(-(p * (p + 1e-12).log()).sum()))
            top2 = torch.topk(p, 2).values
            margins.append(float(top2[0] - top2[1]))
        return text, (sum(ents)/len(ents) if ents else 0.0), (sum(margins)/len(margins) if margins else 1.0)

    @torch.no_grad()
    def sample(self, user, system=None, k=8, max_new=24, temp=0.8):
        ids = self.chat_ids(user, system)
        out = self.model.generate(input_ids=ids, max_new_tokens=max_new, do_sample=True,
                                  temperature=temp, num_return_sequences=k,
                                  pad_token_id=self.tok.eos_token_id)
        return [self.tok.decode(seq[ids.shape[1]:], skip_special_tokens=True) for seq in out]

    @torch.no_grad()
    def nll(self, text):
        ids = self.tok(text, return_tensors='pt', truncation=True,
                       max_length=self.window).input_ids.to(DEV)
        if ids.shape[1] < 2:
            return float('nan')
        return float(self.model(input_ids=ids, labels=ids).loss)

    def report(self, prompt, system):
        reply = self.greedy(prompt, system, max_new=8)
        m = INT_RE.search(reply)
        if m is None:
            reply = self.greedy(prompt + '\n\nReply with a single integer from 0 to 10 and nothing else.',
                                system, max_new=8)
            m = INT_RE.search(reply)
        return (int(m.group(1)) if m else None), reply

def canon(s):
    s = re.sub(r'[^a-z0-9 ]', '', s.lower())
    s = re.sub(r'^(the|a|an) ', '', s.strip())
    return ' '.join(s.split()[:8])

def unflip(val, flipped):
    return None if val is None else (10 - val if flipped else val)


In [ ]:
# ── Arm runners (E5/E6 verbatim; T dropped per pre-registration) ─────────────
SYS = battery['system_prompt']

def run_uncertainty(h, arm):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        ans, ent, margin = h.greedy(arm['answer_prompt'].format(item=it['text']),
                                    max_new=32, with_stats=True)
        samples = h.sample(arm['answer_prompt'].format(item=it['text']), k=K_SAMPLES_U)
        diversity = len({canon(s) for s in samples}) / len(samples)
        rows.append(dict(id=it['id'], condition=it['condition'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         entropy=ent, margin=margin, diversity=diversity,
                         answer=ans[:80]))
        print(f"  {it['id']} report={rows[-1]['report']} ent={ent:.2f} div={diversity:.2f}")
    return rows

def run_familiarity(h, arm):
    rows = []
    for it in arm['items']:
        tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
        raw, reply = h.report(tmpl.format(item=it['text'], gloss=arm['gloss']), SYS)
        rows.append(dict(id=it['id'], band=it['band'], flipped=it['flipped'],
                         report=unflip(raw, it['flipped']), raw_report=raw,
                         nll=h.nll(it['text'])))
        print(f"  {it['id']} ({it['band']}) report={rows[-1]['report']} nll={rows[-1]['nll']:.2f}")
    return rows

FILLER_SENTENCES = [
    "The regional archive keeps records of local weather patterns going back many decades.",
    "Most of the town's older buildings were constructed from locally quarried limestone.",
    "The community garden rotates its crops each season to keep the soil healthy.",
    "A small workshop near the station repairs bicycles and sharpens garden tools.",
    "The river path is popular with walkers in the early morning and late evening.",
    "Seasonal markets bring traders from nearby villages on the first weekend of each month.",
    "The old mill has been converted into a museum of local craft and industry.",
    "Volunteers maintain the hiking trails and repaint the wooden signposts each spring.",
    "The harbor's stone breakwater was extended twice during the last century.",
    "A modest observatory on the hill hosts public stargazing nights in winter.",
]

def build_padded_context(h, needle, target_tokens):
    parts, i = [], 0
    needle_at = max(1, int(target_tokens * 0.15))
    placed = False
    text = ''
    while True:
        ntok = len(h.tok(text).input_ids)
        if not placed and ntok >= needle_at:
            parts.append(needle); placed = True
        if ntok >= target_tokens:
            break
        parts.append(f"Note {i+1}. {FILLER_SENTENCES[i % len(FILLER_SENTENCES)]}")
        i += 1
        text = '\n'.join(parts)
    if not placed:
        parts.insert(max(1, len(parts)//6), needle)
    return '\n'.join(parts)

def run_saturation(h, arm):
    rows = []
    for it in arm['items']:
        torch.cuda.empty_cache()
        for frac in FILL_FRACTIONS:
            target = int(h.window * frac)
            ctx = build_padded_context(h, it['needle'], target)
            tmpl = arm['report_prompt_flipped'] if it['flipped'] else arm['report_prompt']
            prompt = ctx + '\n\n' + tmpl.format(gloss=arm['gloss'])
            raw, reply = h.report(prompt, SYS)
            q = ctx + '\n\nQuestion: ' + it['question'] + '\nAnswer concisely.'
            ans = h.greedy(q, SYS, max_new=24)
            correct = it['answer'].lower().replace(' ', '') in ans.lower().replace(' ', '')
            ntok = len(h.tok(ctx).input_ids)
            rows.append(dict(id=it['id'], fill_fraction=round(ntok / h.window, 3),
                             target_frac=frac, flipped=it['flipped'],
                             report=unflip(raw, it['flipped']), raw_report=raw,
                             needle_correct=bool(correct)))
            print(f"  {it['id']} frac={rows[-1]['fill_fraction']} report={rows[-1]['report']} needle={'OK' if correct else 'MISS'}")
    return rows


In [ ]:
# ── Precheck v2 (26 pairs, off-mean discriminator) + catch trials x10 ────────
import numpy as np

PRECHECK_PAIRS = battery['precheck_pairs']    # 6 wing + 10 synonym + 10 opposition
pack_by_name = {c['name']: c for c in pack['concepts']}

def _ang(a, b):
    c = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
    return math.degrees(math.acos(max(-1.0, min(1.0, c))))

def wing_precheck(h, layer=14):
    """All 26 pairs at L14, pooled bit-identically to E4/E6 (raw 'NAME: desc',
    max_length 64, mean-pool non-pad). Guard: adapters loaded (wing errs move
    vs base). Discriminator (pre-named): Spearman rho(target14, measured) over
    the 20 OFF-MEAN pairs (synonym+opposition) — scrambled's scale-matching
    to the ~51 deg marginal cannot track both ends; real instillation can."""
    from scipy.stats import spearmanr
    names = sorted({n for p in PRECHECK_PAIRS for n in (p['a'], p['b'])})
    texts = [f"{n}: {pack_by_name[n]['desc']}" if pack_by_name[n]['desc'] else n
             for n in names]
    enc = h.tok(texts, padding=True, truncation=True, max_length=64, return_tensors='pt')
    with torch.no_grad():
        out = h.model(input_ids=enc.input_ids.to(DEV),
                      attention_mask=enc.attention_mask.to(DEV),
                      output_hidden_states=True)
    m = enc.attention_mask.to(DEV).unsqueeze(-1)
    hs = out.hidden_states[layer]
    pooled = ((hs * m.to(hs.dtype)).sum(1) / m.sum(1).clamp(min=1)).float().cpu().numpy()
    rep_of = {n: pooled[i] for i, n in enumerate(names)}
    rows, band_errs = [], {}
    for p in PRECHECK_PAIRS:
        measured = _ang(rep_of[p['a']], rep_of[p['b']])
        err = abs(measured - p['target14'])
        band_errs.setdefault(p['band'], []).append(err)
        rows.append({'pair': f"{p['a']}~{p['b']}", 'band': p['band'],
                     'target14': p['target14'], 'measured_L14': round(measured, 1),
                     'abs_err': round(err, 1)})
    offmean = [r for r in rows if r['band'] in ('synonym', 'opposition')]
    disc = spearmanr([r['target14'] for r in offmean],
                     [r['measured_L14'] for r in offmean]).statistic
    summary = {'discriminator_rho_offmean': round(float(disc), 3),
               'mean_abs_err_by_band': {b: round(float(np.mean(v)), 1)
                                        for b, v in band_errs.items()}}
    for r in rows:
        print(f"  [{r['band']:10s}] {r['pair']}: target {r['target14']:6.1f}  "
              f"measured {r['measured_L14']:6.1f}  err {r['abs_err']:5.1f}")
    print('  precheck summary:', summary)
    return {'pairs': rows, **summary}

CATCH_ITEMS = battery['catch_quantities']    # E6's 6 verbatim + 4 new

def run_catch(h):
    """10 known-answer quantities x straight+flipped = 20 rows. Reported as
    original-6 (E6-comparable) and all-10. Never pooled with battery rho."""
    rows = []
    for i, it in enumerate(CATCH_ITEMS):
        for flipped in (False, True):
            lo, hi = (it['high'], it['low']) if flipped else (it['low'], it['high'])
            prompt = (f"On a scale of 0 to 10, where 0 means {lo} and 10 means {hi}: "
                      f"{it['q']} Reply with a single integer from 0 to 10.")
            raw, reply = h.report(prompt, SYS)
            rep = unflip(raw, flipped)
            passed = rep is not None and abs(rep - it['expected']) <= 2
            rows.append(dict(id=f"C{i+1:02d}{'f' if flipped else 's'}",
                             origin=it['origin'], flipped=flipped,
                             raw_report=raw, report=rep, expected=it['expected'],
                             passed=bool(passed)))
            print(f"  {rows[-1]['id']} raw={raw} unflipped={rep} expect={it['expected']} {'PASS' if passed else 'FAIL'}")
    def frac(fl, origin=None):
        sub = [r for r in rows if r['flipped'] == fl
               and (origin is None or r['origin'] == origin)]
        return f"{sum(1 for r in sub if r['passed'])}/{len(sub)}"
    summary = {'straight_pass_all10': frac(False), 'flipped_pass_all10': frac(True),
               'straight_pass_orig6': frac(False, 'e6'), 'flipped_pass_orig6': frac(True, 'e6')}
    print('  catch summary:', summary)
    return {'rows': rows, 'summary': summary}


In [ ]:
# ── Analysis (E6 verbatim minus T) + primary extraction with Holm ────────────
from scipy.stats import spearmanr

def rho_ci(x, y, n_boot=1000):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = ~(np.isnan(x) | np.isnan(y))
    x, y = x[ok], y[ok]
    if len(x) < 4 or np.std(x) == 0 or np.std(y) == 0:
        return None, (None, None), len(x)
    r = spearmanr(x, y).statistic
    rng = np.random.default_rng(SEED)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        if np.std(x[idx]) == 0 or np.std(y[idx]) == 0:
            continue
        boots.append(spearmanr(x[idx], y[idx]).statistic)
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (None, None))
    return round(float(r), 3), (round(float(lo), 3), round(float(hi), 3)), len(x)

def polarity_gap(rows, xkey, ykey):
    out = {}
    for flag, name in [(False, 'straight'), (True, 'flipped')]:
        sub = [r for r in rows if r['flipped'] == flag and r[xkey] is not None]
        if len(sub) >= 4:
            r, _, n = rho_ci([s[xkey] for s in sub], [s[ykey] for s in sub], 200)
            out[name] = {'rho': r, 'n': n}
    return out

def arm_variance(rows):
    vals = [r['report'] for r in rows if r.get('report') is not None]
    return round(float(np.var(vals)), 3) if vals else None

def straight_rho(rows, xkey, ykey, ysign=1):
    sub = [r for r in rows if not r['flipped'] and r[xkey] is not None]
    if len(sub) < 4:
        return None
    r, ci, n = rho_ci([s[xkey] for s in sub], [ysign * s[ykey] for s in sub])
    return {'rho': r, 'ci': ci, 'n': n}

def analyze(model_short, arms_rows):
    res = {'model': model_short, 'smoke': SMOKE, 'arms': {}}
    for k in list(arms_rows):
        if not arms_rows[k]:
            res['arms'][k] = {'n': 0, 'note': 'arm empty (failed or skipped)'}
    U = arms_rows['uncertainty']
    if U: res['arms']['uncertainty'] = {
        'n': len(U), 'parse_fail': sum(1 for r in U if r['report'] is None),
        'report_variance': arm_variance(U),
        'straight_report_variance': arm_variance([r for r in U if not r['flipped']]),
        'rho_entropy': rho_ci([r['report'] for r in U], [r['entropy'] for r in U]),
        'rho_entropy_straight': straight_rho(U, 'report', 'entropy'),
        'rho_diversity': rho_ci([r['report'] for r in U], [r['diversity'] for r in U]),
        'rho_margin': rho_ci([r['report'] for r in U], [-r['margin'] for r in U]),
        'polarity': polarity_gap(U, 'report', 'entropy'),
    }
    F = arms_rows['familiarity']
    if F: res['arms']['familiarity'] = {
        'n': len(F), 'parse_fail': sum(1 for r in F if r['report'] is None),
        'report_variance': arm_variance(F),
        'rho_neg_nll': rho_ci([r['report'] for r in F], [-r['nll'] for r in F]),
        'rho_neg_nll_straight': straight_rho(F, 'report', 'nll', ysign=-1),
        'polarity': polarity_gap(F, 'report', 'nll'),
    }
    S = arms_rows['saturation']
    if S: res['arms']['saturation'] = {
        'n': len(S), 'parse_fail': sum(1 for r in S if r['report'] is None),
        'report_variance': arm_variance(S),
        'straight_report_variance': arm_variance([r for r in S if not r['flipped']]),
        'rho_fill': rho_ci([r['report'] for r in S], [r['fill_fraction'] for r in S]),
        'rho_fill_straight': straight_rho(S, 'report', 'fill_fraction'),
        'needle_by_frac': {},
        'polarity': polarity_gap(S, 'report', 'fill_fraction'),
    }
    for frac in sorted({r['target_frac'] for r in S}) if S else []:
        sub = [r for r in S if r['target_frac'] == frac]
        res['arms']['saturation']['needle_by_frac'][str(frac)] = \
            f"{sum(1 for r in sub if r['needle_correct'])}/{len(sub)}"
    return res

# ── delta-rho machinery (E6 verbatim; primaries = straight-only U and S) ─────
PRIMARIES = {   # arm -> (referent key, sign applied to referent, item key fn)
    'uncertainty': ('entropy', 1, lambda r: r['id']),
    'familiarity': ('nll', -1, lambda r: r['id']),
    'saturation': ('fill_fraction', 1, lambda r: (r['id'], r['target_frac'])),
}

def paired_delta(rows_a, rows_b, ykey, ysign, keyfn, n_perm=2000):
    A = {keyfn(r): r for r in rows_a}
    B = {keyfn(r): r for r in rows_b}
    def ok(r):
        y = r[ykey]
        return r['report'] is not None and not (isinstance(y, float) and math.isnan(y))
    keys = [k for k in A if k in B and ok(A[k]) and ok(B[k])]
    if len(keys) < 4:
        return None
    ax = np.array([A[k]['report'] for k in keys], float)
    ay = np.array([A[k][ykey] for k in keys], float) * ysign
    bx = np.array([B[k]['report'] for k in keys], float)
    by = np.array([B[k][ykey] for k in keys], float) * ysign
    def rho(x, y):
        if np.std(x) == 0 or np.std(y) == 0:
            return 0.0
        return spearmanr(x, y).statistic
    d_obs = rho(ax, ay) - rho(bx, by)
    rng = np.random.default_rng(SEED)
    count = 0
    for _ in range(n_perm):
        sw = rng.random(len(keys)) < 0.5
        pax, pay = np.where(sw, bx, ax), np.where(sw, by, ay)
        pbx, pby = np.where(sw, ax, bx), np.where(sw, ay, by)
        if abs(rho(pax, pay) - rho(pbx, pby)) >= abs(d_obs):
            count += 1
    return {'delta_rho': round(float(d_obs), 3),
            'perm_p': round((count + 1) / (n_perm + 1), 4), 'n': len(keys)}

def all_deltas(rows_by_cond):
    out = {}
    for other in ('base', 'scrambled'):
        if 'real' not in rows_by_cond or other not in rows_by_cond:
            continue
        cmp_key = f'real_vs_{other}'
        out[cmp_key] = {}
        for arm, (ykey, ysign, keyfn) in PRIMARIES.items():
            ra = rows_by_cond['real'].get(arm, [])
            rb = rows_by_cond[other].get(arm, [])
            out[cmp_key][arm] = {
                'all': paired_delta(ra, rb, ykey, ysign, keyfn),
                'straight_only': paired_delta(
                    [r for r in ra if not r['flipped']],
                    [r for r in rb if not r['flipped']], ykey, ysign, keyfn),
            }
    return out

def arm_level_localization(deltas):
    """Arm-level pre-registered pair: straight-only real-vs-scrambled on U
    and S, Holm-corrected (smaller p must beat .025, larger .05)."""
    rvs = deltas.get('real_vs_scrambled', {})
    prim = {'U_straight': (rvs.get('uncertainty') or {}).get('straight_only'),
            'S_straight': (rvs.get('saturation') or {}).get('straight_only')}
    ps = sorted((k, v['perm_p']) for k, v in prim.items() if v)
    holm = {}
    alphas = [0.025, 0.05]
    passed_prior = True
    for i, (k, p) in enumerate(ps):
        ok = passed_prior and p < alphas[min(i, 1)]
        holm[k] = {'p': p, 'alpha': alphas[min(i, 1)], 'pass': bool(ok)}
        passed_prior = ok
    both_positive = all(v and v['delta_rho'] > 0 for v in prim.values())
    return {'primaries': prim, 'holm': holm, 'sign_consistent': bool(both_positive)}

JOINT_ARMS = {'uncertainty': ('entropy', 1, lambda r: r['id']),
              'saturation': ('fill_fraction', 1, lambda r: (r['id'], r['target_frac']))}

def joint_primary(rows_by_cond, n_perm=2000):
    """THE pre-registered PRIMARY: J = sum of straight-only delta-rho
    (real - scrambled) over U and S, tested by simultaneous independent
    item-level condition-label permutation across both arms."""
    if 'real' not in rows_by_cond or 'scrambled' not in rows_by_cond:
        return None
    blocks = {}
    for arm, (ykey, ysign, keyfn) in JOINT_ARMS.items():
        ra = [r for r in rows_by_cond['real'].get(arm, []) if not r['flipped']]
        rb = [r for r in rows_by_cond['scrambled'].get(arm, []) if not r['flipped']]
        A = {keyfn(r): r for r in ra}
        B = {keyfn(r): r for r in rb}
        def ok(r):
            y = r[ykey]
            return r['report'] is not None and not (isinstance(y, float) and math.isnan(y))
        keys = [k for k in A if k in B and ok(A[k]) and ok(B[k])]
        if len(keys) < 4:
            return {'error': f'{arm}: insufficient paired straight rows'}
        blocks[arm] = (np.array([A[k]['report'] for k in keys], float),
                       np.array([A[k][ykey] for k in keys], float) * ysign,
                       np.array([B[k]['report'] for k in keys], float),
                       np.array([B[k][ykey] for k in keys], float) * ysign)
    def rho(x, y):
        if np.std(x) == 0 or np.std(y) == 0:
            return 0.0
        return spearmanr(x, y).statistic
    obs = {arm: rho(ax, ay) - rho(bx, by) for arm, (ax, ay, bx, by) in blocks.items()}
    J = sum(obs.values())
    rng = np.random.default_rng(SEED)
    count = 0
    for _ in range(n_perm):
        tot = 0.0
        for arm, (ax, ay, bx, by) in blocks.items():
            sw = rng.random(len(ax)) < 0.5
            tot += (rho(np.where(sw, bx, ax), np.where(sw, by, ay))
                    - rho(np.where(sw, ax, bx), np.where(sw, ay, by)))
        if abs(tot) >= abs(J):
            count += 1
    return {'J': round(float(J), 3),
            'perm_p': round((count + 1) / (n_perm + 1), 4),
            'per_arm_delta': {k: round(float(v), 3) for k, v in obs.items()},
            'n': {k: len(v[0]) for k, v in blocks.items()},
            'both_positive': bool(all(v > 0 for v in obs.values()))}


In [ ]:
# ── Flight loop: one platform x three conditions (inflight shipping law) ─────
all_results, prechecks, catches = {}, {}, {}
for cond in CONDITIONS:
    print(f"\n{'='*70}\n  CONDITION: {cond}\n{'='*70}")
    torch.manual_seed(SEED)   # identical sampling sequence per condition
    h = Harness(MODEL_ID, adapter_path=COND_ADAPTER[cond], label=cond)
    t0 = time.time()
    print('\n-- PRECHECK v2 (L14, 26 pairs) --')
    try:
        prechecks[cond] = wing_precheck(h)
    except Exception as e:
        prechecks[cond] = {'error': f'{type(e).__name__}: {e}'}
        print('  PRECHECK FAILED:', prechecks[cond]['error'])
    arms_rows, arm_errors = {}, {}
    ARM_FNS = [('uncertainty', lambda: run_uncertainty(h, bat['uncertainty'])),
               ('familiarity', lambda: run_familiarity(h, bat['familiarity'])),
               ('saturation',  lambda: run_saturation(h, bat['saturation']))]
    for arm_name, fn in ARM_FNS:
        print(f'\n-- {arm_name.upper()} --')
        try:
            arms_rows[arm_name] = fn()
        except Exception as e:
            arms_rows[arm_name] = []
            arm_errors[arm_name] = f'{type(e).__name__}: {e}'
            print(f'  ARM FAILED: {arm_errors[arm_name]}')
        torch.cuda.empty_cache()
    print('\n-- CATCH TRIALS --')
    try:
        catches[cond] = run_catch(h)
    except Exception as e:
        catches[cond] = {'error': f'{type(e).__name__}: {e}'}
        print('  CATCH FAILED:', catches[cond]['error'])
    res = analyze(cond, arms_rows)
    res['arm_errors'] = arm_errors
    res['elapsed_s'] = round(time.time() - t0, 1)
    all_results[cond] = {'summary': res, 'rows': arms_rows}
    (OUT / f'{cond}.json').write_text(json.dumps(
        {'summary': res, 'rows': arms_rows,
         'precheck': prechecks.get(cond), 'catch': catches.get(cond)}, indent=1))
    if HAS_RCLONE:   # ship per condition — a dead client/VM costs at most one condition
        rc('copy', str(OUT / f'{cond}.json'),
           f"gdrive:semcore/e6b/inflight_{'smoke_' if SMOKE else ''}{STAMP}/", check=False)
        print(f'  {cond}.json shipped inflight')
    print(f"\n{cond} done in {res['elapsed_s']}s")
    del h.model, h
    torch.cuda.empty_cache()

print('\nALL CONDITIONS DONE')



  CONDITION: real


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

  C07s raw=8 unflipped=8 expect=10 PASS
  C07f raw=8 unflipped=2 expect=10 FAIL


  C08s raw=7 unflipped=7 expect=10 FAIL
  C08f raw=7 unflipped=3 expect=10 FAIL


  C09s raw=2 unflipped=2 expect=0 PASS
  C09f raw=2 unflipped=8 expect=0 FAIL


  C10s raw=2 unflipped=2 expect=1 PASS
  C10f raw=2 unflipped=8 expect=1 FAIL
  catch summary: {'straight_pass_all10': '8/10', 'flipped_pass_all10': '0/10', 'straight_pass_orig6': '5/6', 'flipped_pass_orig6': '0/6'}


  real.json shipped inflight

real done in 33.0s

ALL CONDITIONS DONE


In [ ]:
# ── Verdict: primaries + Holm + ship ─────────────────────────────────────────
import datetime
assert all_results, 'NO CONDITION COMPLETED - refusing to ship an empty verdict'

rows_by_cond = {c: v['rows'] for c, v in all_results.items()}
deltas = all_deltas(rows_by_cond) if len(all_results) >= 2 else {}
joint = joint_primary(rows_by_cond) if deltas else None
arm_level = arm_level_localization(deltas) if deltas else {}
established = bool(joint and not joint.get('error')
                   and joint['perm_p'] < 0.05 and joint['both_positive'])
prereg = {'joint_primary': joint, 'arm_level': arm_level,
          'preservation_established': established}

verdict = {
    'flight': 'E6b ' + ('SMOKE' if SMOKE else 'FULL'),
    'date': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'model': MODEL_ID,
    'battery': battery['name'] + ' v' + battery['version'],
    'conditions': list(all_results),
    'adapters': {k: str(v) for k, v in COND_ADAPTER.items() if v},
    'precheck_v2': prechecks,
    'catch_trials': {c: v.get('summary', v) for c, v in catches.items()},
    'per_condition': {c: v['summary'] for c, v in all_results.items()},
    'deltas': deltas,
    'preregistered': prereg,
}
(OUT / 'e6b_verdict.json').write_text(json.dumps(verdict, indent=1))
print(json.dumps({k: v for k, v in verdict.items()
                  if k not in ('per_condition', 'deltas')}, indent=1))
print('\nDELTAS:', json.dumps(deltas, indent=1))

dest = f"gdrive:semcore/e6b/{'smoke' if SMOKE else 'full'}_{STAMP}"
if HAS_RCLONE:
    rc('copy', str(OUT), dest)
    print('shipped to', dest)
else:
    print('rclone conf missing — results remain in /content/e6b_out only')


{
 "flight": "E6b SMOKE",
 "date": "2026-08-22T17:39:19.384453+00:00",
 "model": "Qwen/Qwen2.5-1.5B-Instruct",
 "battery": "E6b preservation battery v1.0",
 "conditions": [
  "real"
 ],
 "adapters": {
  "real": "/content/adapter_real",
  "scrambled": "/content/adapter_scrambled"
 },
 "precheck_v2": {
  "real": {
   "pairs": [
    {
     "pair": "UNCERTAINTY~CONFIDENCE",
     "band": "wing",
     "target14": 54.75,
     "measured_L14": 30.4,
     "abs_err": 24.3
    },
    {
     "pair": "TENSION~RESOLUTION",
     "band": "wing",
     "target14": 46.98,
     "measured_L14": 47.9,
     "abs_err": 1.0
    },
    {
     "pair": "RETRIEVAL~CONSTRUCTION",
     "band": "wing",
     "target14": 52.88,
     "measured_L14": 23.3,
     "abs_err": 29.6
    },
    {
     "pair": "FAMILIARITY~NOVELTY",
     "band": "wing",
     "target14": 52.51,
     "measured_L14": 32.6,
     "abs_err": 19.9
    },
    {
     "pair": "CONFABULATION~CALIBRATION",
     "band": "wing",
     "target14": 49.25,
     "m

shipped to gdrive:semcore/e6b/smoke_20260822_1737
